# Final Bicubic Protocol — Set5 x2 Pilot

Run every cell from top to bottom in Google Colab. This notebook validates the prepared Set5 HR/LR pairs, runs only the x2 bicubic pilot, and saves new results without touching the preliminary CSV files.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/divinesta/SuperResolution-Comparative-Analysis.git"
REPO_ROOT = Path("/content/SuperResolution-Comparative-Analysis")

if REPO_ROOT.exists():
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"Repository ready: {REPO_ROOT}")

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
    check=True,
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Dependencies installed.")

In [ ]:
from datetime import UTC, datetime

DATA_ROOT = Path("/content/drive/MyDrive/FYP_SR_Data")
RUN_ID = datetime.now(UTC).strftime("%Y%m%d_%H%M%S_utc")
PILOT_ROOT = DATA_ROOT / "results" / "final_bicubic" / "pilot" / RUN_ID
PILOT_CSV = PILOT_ROOT / "Set5_x2_bicubic_pilot.csv"
PILOT_SR_DIR = PILOT_ROOT / "reconstructions"

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")

print(f"Data root: {DATA_ROOT}")
print(f"This pilot will save to: {PILOT_ROOT}")

In [ ]:
from app.evaluation.data_validation import validate_prepared_dataset

validation = validate_prepared_dataset("Set5", 2, DATA_ROOT)
print(
    f"VALID: {validation.dataset} x{validation.scale} has "
    f"{validation.image_count} complete HR/LR pairs."
)
print(f"HR folder: {validation.hr_directory}")
print(f"LR folder: {validation.lr_directory}")

In [ ]:
from app.evaluation.bicubic import (
    BicubicEvaluationConfig,
    evaluate_bicubic_dataset,
    write_results_csv,
)

config = BicubicEvaluationConfig(
    dataset="Set5",
    scale=2,
    warmup_runs=3,
    timed_runs=10,
)
records = evaluate_bicubic_dataset(
    validation.hr_directory,
    validation.lr_directory,
    config,
    sr_output_dir=PILOT_SR_DIR,
)
write_results_csv(records, PILOT_CSV)

print(f"Evaluated {len(records)} Set5 images at x2.")
print(f"CSV saved to: {PILOT_CSV}")
print(f"Reconstructions saved to: {PILOT_SR_DIR}")

In [ ]:
from statistics import mean

summary_fields = (
    "psnr_y",
    "ssim_y",
    "psnr_rgb",
    "ssim_rgb",
    "latency_mean_ms",
)
summary = {
    field: mean(float(record[field]) for record in records)
    for field in summary_fields
}

print("SET5 x2 BICUBIC PILOT SUMMARY")
print(f"Images: {len(records)}")
print(f"Y PSNR: {summary['psnr_y']:.4f} dB")
print(f"Y SSIM: {summary['ssim_y']:.4f}")
print(f"RGB PSNR: {summary['psnr_rgb']:.4f} dB")
print(f"RGB SSIM: {summary['ssim_rgb']:.4f}")
print(f"Mean latency: {summary['latency_mean_ms']:.4f} ms")
print(f"Saved CSV: {PILOT_CSV}")

## Stop after the pilot

Send the printed summary and any error output back for review. Do not start the full 12-combination run until this pilot is checked.